# 1. Mount Google Drive (Optional)
Mount Google Drive to persist models, outputs, or custom nodes across sessions.

In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive (Optional)</h3></center><br>
MODE = "SKIP" #@param ["MOUNT", "UNMOUNT", "SKIP"]

from google.colab import drive
import os

if MODE == "MOUNT":
    drive.mount._DEBUG = False
    drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
    try:
        drive.flush_and_unmount()
    except ValueError:
        pass
    get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")
else:
    print("Skipping Google Drive mount.")


# 2. Setup ComfyUI, Dependencies & Custom Nodes (T4 / GGUF - Zero Restarts)
Clones latest ComfyUI (with native MiniMax H3 support), installs core & GGUF packages (`requirements.txt`, `torchao`, `accelerate`, `gguf`, `sentencepiece`, `protobuf`, `gradio`), and pre-installs `ComfyUI-Manager`, `ComfyUI-H3-Modality-Lora-Loader`, `ComfyUI-MiniMax-H3-Turbo`, `ComfyUI-GGUF`, and `ComfyUI-H3-Multishot` (`H3ClipLoaderAny` + `mmproj` support) so no server restarts are needed.

In [ ]:
#@markdown ### Setup ComfyUI & Core Components (T4 / GGUF Edition)
import os
from pathlib import Path

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  # @param {type:"boolean"}
PULL_GITHUB_CHANGES = True  # @param {type:"boolean"}
INSTALL_COMFY_MANAGER = True  # @param {type:"boolean"}
INSTALL_H3_LORA_LOADER = True  # @param {type:"boolean"}
INSTALL_GGUF_NODES = True  # @param {type:"boolean"}

# Pull latest changes from ComfyUI-On-Colab repo or clone if missing
if PULL_GITHUB_CHANGES:
    repo_candidates = ["/content/ComfyUI-On-Colab", "/content/drive/MyDrive/ComfyUI-On-Colab"]
    repo_found = False
    for r_dir in repo_candidates:
        if os.path.exists(os.path.join(r_dir, ".git")):
            print(f"-= Pulling latest changes in {r_dir} =-")
            !git -C {r_dir} pull
            repo_found = True
            break
    if not repo_found:
        print("-= Cloning ComfyUI-On-Colab repo to /content/ComfyUI-On-Colab =-")
        !git clone https://github.com/parsamrrelax/ComfyUI-On-Colab.git /content/ComfyUI-On-Colab

WORKSPACE = "/content/ComfyUI"
if DRIVE_PATH.strip():
    WORKSPACE = os.path.join(DRIVE_PATH.strip(), "ComfyUI")

%cd /content
if not os.path.exists(WORKSPACE):
    print("-= Initial setup: Cloning ComfyUI =-")
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}

%cd {WORKSPACE}

if UPDATE_COMFY_UI:
    print("-= Updating ComfyUI to latest version =-")
    !git pull

print("-= Installing Core & GGUF Dependencies =-")
!pip install -q -r requirements.txt
!pip install -q torchao accelerate xformers gradio
!pip install -q --upgrade gguf sentencepiece protobuf

# Create required model and workflow directories for MiniMax H3
os.makedirs(f"{WORKSPACE}/models/diffusion_models", exist_ok=True)
os.makedirs(f"{WORKSPACE}/models/unet", exist_ok=True)
os.makedirs(f"{WORKSPACE}/models/text_encoders", exist_ok=True)
os.makedirs(f"{WORKSPACE}/models/vae", exist_ok=True)
os.makedirs(f"{WORKSPACE}/models/checkpoints", exist_ok=True)
os.makedirs(f"{WORKSPACE}/models/loras", exist_ok=True)
os.makedirs(f"{WORKSPACE}/user/default/workflows", exist_ok=True)
os.makedirs("/content/workflows", exist_ok=True)

def clone_or_update(repo_url, dest_dir, label):
    if not os.path.exists(dest_dir):
        print(f"-= Installing {label} =-")
        !git clone {repo_url} {dest_dir}
    else:
        print(f"-= Updating {label} =-")
        !git -C {dest_dir} pull
    req_file = os.path.join(dest_dir, "requirements.txt")
    if os.path.exists(req_file):
        print(f"-= Installing requirements for {label} =-")
        !pip install -q -r {req_file}

# Pre-install ComfyUI-Manager
if INSTALL_COMFY_MANAGER:
    clone_or_update("https://github.com/ltdrdata/ComfyUI-Manager.git", f"{WORKSPACE}/custom_nodes/ComfyUI-Manager", "ComfyUI-Manager")

# Pre-install H3 LoRA & Turbo nodes
if INSTALL_H3_LORA_LOADER:
    clone_or_update("https://github.com/Dantemss/ComfyUI-H3-Modality-Lora-Loader.git", f"{WORKSPACE}/custom_nodes/ComfyUI-H3-Modality-Lora-Loader", "ComfyUI-H3-Modality-Lora-Loader")
    clone_or_update("https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git", f"{WORKSPACE}/custom_nodes/ComfyUI-MiniMax-H3-Turbo", "ComfyUI-MiniMax-H3-Turbo")

# Pre-install GGUF & H3-Multishot nodes (required for GGUF diffusion + Qwen3-VL mmproj loading without restart)
if INSTALL_GGUF_NODES:
    clone_or_update("https://github.com/city96/ComfyUI-GGUF.git", f"{WORKSPACE}/custom_nodes/ComfyUI-GGUF", "ComfyUI-GGUF")
    clone_or_update("https://github.com/jlucasmcrell/ComfyUI-H3-Multishot.git", f"{WORKSPACE}/custom_nodes/ComfyUI-H3-Multishot", "ComfyUI-H3-Multishot")

print("\n✅ ComfyUI T4 GGUF setup and all custom node dependencies are ready (zero restarts needed)!")


# 3. Download MiniMax H3 12GB/T4 Workflows (GGUF + Multi-LoRA)
Downloads **`minimax_h3_image_12gb_with_lora.json`** (configured for GGUF diffusion + GGUF Qwen3-VL + BF16 mmproj + Tiled VAE Decode + 3 chained LoRA slots + 4 MP default) along with the base 12GB/8GB workflows into `ComfyUI/user/default/workflows/` and `/content/workflows/`.

In [ ]:
#@markdown ### Download MiniMax H3 12GB / T4 Workflows (with LoRA Support)
import os
import shutil
if 'WORKSPACE' not in globals():
    WORKSPACE = '/content/ComfyUI'

workflows = {
    "minimax_h3_image_12gb_with_lora.json": "https://raw.githubusercontent.com/parsamrrelax/ComfyUI-On-Colab/main/minimax_h3_image_12gb_with_lora.json",
    "minimax_h3_image_12gb_vram.json": "https://huggingface.co/vyrekai/minimax-h3-image-comfyui-workflows/resolve/main/minimax_h3_image_12gb_vram.json?download=true",
    "minimax_h3_image_8gb_vram.json": "https://huggingface.co/vyrekai/minimax-h3-image-comfyui-workflows/resolve/main/minimax_h3_image_8gb_vram.json?download=true"
}

target_dirs = [
    f"{WORKSPACE}/user/default/workflows",
    "/content/workflows"
]

local_repo_candidates = ["/content/ComfyUI-On-Colab", "/content/drive/MyDrive/ComfyUI-On-Colab"]
local_repo_wf = None
for cand in local_repo_candidates:
    cand_wf = os.path.join(cand, "minimax_h3_image_12gb_with_lora.json")
    if os.path.exists(cand_wf):
        local_repo_wf = cand_wf
        break

for name, url in workflows.items():
    for t_dir in target_dirs:
        os.makedirs(t_dir, exist_ok=True)
        dest = os.path.join(t_dir, name)
        if name == "minimax_h3_image_12gb_with_lora.json" and local_repo_wf:
            shutil.copy2(local_repo_wf, dest)
        else:
            !wget -q -nc -O "{dest}" "{url}"
    print(f"✅ Installed workflow: {name}")

print("\n🎉 All 12GB/T4 MiniMax H3 workflows are installed!")


# 4. Download MiniMax H3 GGUF Models & LoRAs (Optimized for T4 + 12GB RAM + Face Reference)
Downloads the smaller GGUF models tailored for Colab Free Tier (T4 16GB VRAM / 12GB System RAM) while preserving face identity:
- **Diffusion Model**: `MiniMax-H3-Image-AD-Q4_K.gguf` (~11.6 GB) — Accuracy-Driven 4-bit K-quant for crisp face/skin details.
- **Text/Vision Encoder**: `Qwen3-VL-32B-Instruct-MiniMax-H3-L0-49-UD-Q2_K_XL.gguf` (~9.57 GB) + `Qwen3-VL-32B-Instruct-MiniMax-H3-L0-49-mmproj-BF16.gguf` (~1.2 GB) — keeps vision projection in full BF16 for accurate reference face conditioning.
- **Prompt Upsampler**: Disabled by default (`DOWNLOAD_PROMPT_UPSAMPLER = False`).

In [ ]:
#@markdown ### MiniMax H3 T4 / GGUF Models Download Configuration
DOWNLOAD_DIFFUSION_MODEL = True  # @param {type:"boolean"}
#@markdown > **Diffusion Model (GGUF)**: MiniMax-H3-Image-AD-Q4_K.gguf (~11.6 GB) -> `models/diffusion_models/`

DOWNLOAD_TEXT_ENCODER = True  # @param {type:"boolean"}
#@markdown > **Text/Vision Encoder (GGUF + BF16 mmproj)**: Qwen3-VL-32B-Instruct-MiniMax-H3-L0-49-UD-Q2_K_XL.gguf (~9.57 GB) + mmproj-BF16 (~1.2 GB) -> `models/text_encoders/`

DOWNLOAD_VAE = True  # @param {type:"boolean"}
#@markdown > **VAE**: minimax_h3_image_vae_fp16.safetensors (~1.2 GB) -> `models/vae/`

DOWNLOAD_PROMPT_UPSAMPLER = False  # @param {type:"boolean"}
#@markdown > **Prompt Upsampler (Optional, Off by default)**: qwen3.5_9b_w4a8_gptq.safetensors (~7.3 GB) -> `models/text_encoders/`

DOWNLOAD_SAMPLE_LORA = False  # @param {type:"boolean"}
#@markdown > **Sample LoRA (Optional)**: fal MiniMax H3 Realism People LoRA -> `models/loras/`

DOWNLOAD_H3_LORAS = True  # @param {type:"boolean"}
#@markdown > **H3 LoRAs**: `largebreastsh3.safetensors`, `femaleanatomyh3.safetensors` -> `models/loras/`

CUSTOM_LORA_URL = ""  # @param {type:"string"}
#@markdown > **Custom LoRA URL (Optional)**: Paste any Hugging Face / Civitai direct URL for MiniMax H3 video/image LoRA

import os
if 'WORKSPACE' not in globals():
    WORKSPACE = '/content/ComfyUI'

# Install aria2 for high-speed multi-threaded downloads
!apt-get -y install -qq aria2

def aria_download(url, dest_dir, filename):
    os.makedirs(dest_dir, exist_ok=True)
    target_path = os.path.join(dest_dir, filename)
    if os.path.exists(target_path) and os.path.getsize(target_path) > 1024 * 1024:
        print(f"⏩ {filename} already exists ({os.path.getsize(target_path)/(1024**3):.2f} GB), skipping download.")
        return
    print(f"🚀 Downloading {filename} to {dest_dir}...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{dest_dir}" -o "{filename}"
    print(f"✅ Finished: {filename}")

# 1. MiniMax H3 Image Diffusion Model (GGUF AD-Q4_K - ~11.6 GB)
if DOWNLOAD_DIFFUSION_MODEL:
    aria_download(
        "https://huggingface.co/AtomicChat/MiniMax-H3-Image-GGUF/resolve/main/MiniMax-H3-Image-AD-Q4_K.gguf?download=true",
        f"{WORKSPACE}/models/diffusion_models",
        "MiniMax-H3-Image-AD-Q4_K.gguf"
    )

# 2. Qwen3-VL 32B MiniMax H3 Text/Vision Encoder (GGUF UD-Q2_K_XL ~9.57 GB + BF16 mmproj ~1.2 GB)
if DOWNLOAD_TEXT_ENCODER:
    aria_download(
        "https://huggingface.co/nif0/Qwen3-VL-32B-Instruct-MiniMax-H3-GGUF/resolve/main/Qwen3-VL-32B-Instruct-MiniMax-H3-L0-49-UD-Q2_K_XL.gguf?download=true",
        f"{WORKSPACE}/models/text_encoders",
        "Qwen3-VL-32B-Instruct-MiniMax-H3-L0-49-UD-Q2_K_XL.gguf"
    )
    aria_download(
        "https://huggingface.co/nif0/Qwen3-VL-32B-Instruct-MiniMax-H3-GGUF/resolve/main/Qwen3-VL-32B-Instruct-MiniMax-H3-L0-49-mmproj-BF16.gguf?download=true",
        f"{WORKSPACE}/models/text_encoders",
        "Qwen3-VL-32B-Instruct-MiniMax-H3-L0-49-mmproj-BF16.gguf"
    )

# 3. MiniMax H3 VAE (FP16)
if DOWNLOAD_VAE:
    aria_download(
        "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/vae/minimax_h3_image_vae_fp16.safetensors?download=true",
        f"{WORKSPACE}/models/vae",
        "minimax_h3_image_vae_fp16.safetensors"
    )

# 4. Optional Qwen3.5 9B Prompt Upsampler (Off by default)
if DOWNLOAD_PROMPT_UPSAMPLER:
    aria_download(
        "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/text_encoders/qwen3.5_9b_w4a8_gptq.safetensors?download=true",
        f"{WORKSPACE}/models/text_encoders",
        "qwen3.5_9b_w4a8_gptq.safetensors"
    )

# 5. Sample MiniMax H3 Realism LoRA
if DOWNLOAD_SAMPLE_LORA:
    aria_download(
        "https://huggingface.co/fal/MiniMax-H3-Realism-People-LoRA/resolve/main/minimax_h3_realism_lora.safetensors",
        f"{WORKSPACE}/models/loras",
        "minimax_h3_realism_lora.safetensors"
    )

# 6. Additional H3 LoRAs
if DOWNLOAD_H3_LORAS:
    h3_loras = [
        ("largebreastsh3.safetensors", "https://huggingface.co/Mrrelax/pp/resolve/main/largebreastsh3.safetensors?download=true"),
        ("femaleanatomyh3.safetensors", "https://huggingface.co/Mrrelax/pp/resolve/main/femaleanatomyh3.safetensors?download=true"),
    ]
    for lora_name, lora_url in h3_loras:
        aria_download(lora_url, f"{WORKSPACE}/models/loras", lora_name)

# 7. Custom LoRA URL (e.g. MiniMax H3 video/image LoRA)
if CUSTOM_LORA_URL.strip():
    custom_name = CUSTOM_LORA_URL.strip().split('/')[-1].split('?')[0]
    aria_download(
        CUSTOM_LORA_URL.strip(),
        f"{WORKSPACE}/models/loras",
        custom_name
    )

print("\n🎉 MiniMax H3 T4 GGUF models and LoRAs are completely downloaded and verified!")


# (Optional) Download Extra Checkpoints / CivitAI Models
Optional helper if you wish to download additional models or CivitAI checkpoints into `models/checkpoints/`.

In [ ]:
#@markdown ### Optional: Download Extra Checkpoints
DOWNLOAD_EXTRA_MODELS = False  # @param {type:"boolean"}

if DOWNLOAD_EXTRA_MODELS:
    if 'WORKSPACE' not in globals():
        WORKSPACE = '/content/ComfyUI'
    from google.colab import userdata
    try:
        CIVITAI_API_TOKEN = userdata.get("CIVITAI_API_TOKEN")
        print("Loaded CivitAI API key:", "✅" if CIVITAI_API_TOKEN else "❌ Not found")
    except Exception:
        CIVITAI_API_TOKEN = None

    def downloadCheckpoint(url, filename=None):
        dest = f"{WORKSPACE}/models/checkpoints"
        if "huggingface.co" in url:
            if not filename:
                filename = url.split("/")[-1].removesuffix("?download=true")
            !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{dest}" -o "{filename}"
        else:
            token_suffix = f"?token={CIVITAI_API_TOKEN}" if CIVITAI_API_TOKEN else ""
            full_url = f"{url}{token_suffix}"
            if filename:
                !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{full_url}" -d "{dest}" -o "{filename}"
            else:
                !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{full_url}" -d "{dest}"

    # Example: downloadCheckpoint("https://civitai.com/api/download/models/2071650", "cyberrealisticPony.safetensors")
else:
    print("Skipping extra checkpoints download.")


# 5. Verify File Structure & Available Models
Inspect the installed model files and available workflows before starting the server.

In [ ]:
if 'WORKSPACE' not in globals():
    WORKSPACE = '/content/ComfyUI'

print("=== Diffusion Models (models/diffusion_models/) ===")
!ls -lh {WORKSPACE}/models/diffusion_models/

print("\n=== Text Encoders & mmproj (models/text_encoders/) ===")
!ls -lh {WORKSPACE}/models/text_encoders/

print("\n=== VAE Models (models/vae/) ===")
!ls -lh {WORKSPACE}/models/vae/

print("\n=== LoRA Models (models/loras/) ===")
!ls -lh {WORKSPACE}/models/loras/

print("\n=== Custom Nodes (custom_nodes/) ===")
!ls -1 {WORKSPACE}/custom_nodes/

print("\n=== Workflows (user/default/workflows/) ===")
!ls -lh {WORKSPACE}/user/default/workflows/


# 6. Start ComfyUI & Expose Server (High-Speed Tunnels)
Starts ComfyUI and exposes it publicly.
- **Gradio (Recommended)**: High bandwidth and fast streaming — eliminates the slow image download speeds encountered with Cloudflare free tunnels.
- **Cloudflare**: Standard quick tunnel (`trycloudflare.com`).
- **Pinggy**: High-speed SSH tunnel (`pinggy.link`).
- **Ngrok**: Commercial tunnel (requires `NGROK_TOKEN`).

In [ ]:
#@markdown ### Start ComfyUI with High-Speed Tunnel
TUNNEL_TYPE = "Gradio" #@param ["Gradio", "Cloudflare", "Pinggy", "Ngrok"]
#@markdown > **Gradio (Recommended)**: Fast, unthrottled downloads for large (16MP) images without token setup.
#@markdown > **Cloudflare**: Standard `trycloudflare.com` quick tunnel.
#@markdown > **Pinggy**: Free SSH-based high-speed tunnel (`pinggy.link`).
#@markdown > **Ngrok**: Dedicated tunnel (paste token below if selected).
NGROK_TOKEN = "" #@param {type:"string"}

import subprocess
import threading
import time
import socket
import re
import os
import secrets

if 'WORKSPACE' not in globals():
    WORKSPACE = '/content/ComfyUI'

%cd {WORKSPACE}

# Ensure tunneling packages are installed based on selection
if TUNNEL_TYPE == "Gradio":
    try:
        import gradio
    except ImportError:
        print("Installing gradio for high-speed tunneling...")
        !pip install -q gradio
elif TUNNEL_TYPE == "Ngrok":
    try:
        import pyngrok
    except ImportError:
        print("Installing pyngrok...")
        !pip install -q pyngrok

def wait_for_port(port=8188):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(("127.0.0.1", port))
        sock.close()
        if result == 0:
            break

def print_banner(tunnel_name, url):
    print("\n" + "=" * 75)
    print(f"🚀 ComfyUI {tunnel_name} URL: {url}")
    print("=" * 75)
    print("💡 Tips:")
    print(" - In ComfyUI, click 'Workflows' -> 'Browse Templates' or load:")
    print("   minimax_h3_image_12gb_with_lora.json (12GB/T4 GGUF + Multi-LoRA + 4MP Default)")
    print("   minimax_h3_image_12gb_vram.json (12GB / mid VRAM)")
    print("   minimax_h3_image_8gb_vram.json (8GB / low VRAM)")
    print(" - LoRA nodes are bypassed by default (Ctrl+B on the LoRA node to activate it).\n")

def launch_cloudflare(port=8188):
    if not os.path.exists("/usr/local/bin/cloudflared"):
        print("Installing Cloudflared binary...")
        !curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
        !chmod +x /usr/local/bin/cloudflared

    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    tunnel_url = None
    for line in p.stderr:
        if not tunnel_url:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                tunnel_url = match.group(0)
                print_banner("Cloudflare", tunnel_url)

def create_gradio_tunnel(setup_tunnel, host, port, share_token):
    # share_server_address and share_server_tls_certificate must be None for default gradio.live
    attempts = [
        lambda: setup_tunnel(host, port, share_token, None, None),
        lambda: setup_tunnel(host, port, share_token, None),
        lambda: setup_tunnel(host, port, share_token),
    ]
    last_err = None
    for attempt in attempts:
        try:
            return attempt()
        except (TypeError, ValueError) as e:
            last_err = e
            continue
    raise last_err

def launch_tunnel(tunnel_type, port=8188):
    wait_for_port(port)
    print(f"\n[Tunnel] ComfyUI server is up! Establishing {tunnel_type} tunnel...")

    if tunnel_type == "Gradio":
        try:
            try:
                from gradio.networking import setup_tunnel
            except ImportError:
                try:
                    from gradio.tunneling import setup_tunnel
                except ImportError:
                    from gradio.blocks import setup_tunnel
            share_token = secrets.token_urlsafe(32)
            public_url = create_gradio_tunnel(setup_tunnel, "127.0.0.1", port, share_token)
            print_banner("Gradio (High-Speed)", public_url)
            while True:
                time.sleep(1)
        except Exception as e:
            print(f"\n⚠️ Gradio tunnel could not start ({e}). Falling back to Cloudflare Tunnel...")
            launch_cloudflare(port)

    elif tunnel_type == "Cloudflare":
        launch_cloudflare(port)
    elif tunnel_type == "Pinggy":
        p = subprocess.Popen(
            ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "-p", "443", f"-R0:localhost:{port}", "a.pinggy.io"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1
        )
        tunnel_url = None
        for line in p.stdout:
            if not tunnel_url:
                match = re.search(r"https://[a-zA-Z0-9-]+\.a\.pinggy\.link", line)
                if match:
                    tunnel_url = match.group(0)
                    print_banner("Pinggy (High-Speed)", tunnel_url)

    elif tunnel_type == "Ngrok":
        from pyngrok import ngrok
        if NGROK_TOKEN.strip():
            ngrok.set_auth_token(NGROK_TOKEN.strip())
        public_url = ngrok.connect(port).public_url
        print_banner("Ngrok (High-Speed)", public_url)
        while True:
            time.sleep(1)

threading.Thread(target=launch_tunnel, daemon=True, args=(TUNNEL_TYPE, 8188)).start()

# Run ComfyUI server
!python main.py --listen 127.0.0.1 --port 8188 --dont-print-server
